# Heston-Based Portfolio VaR Research Workflow

This notebook demonstrates the project end to end: portfolio setup, VaR reporting, Monte Carlo path visualization, terminal P&L analysis, and Python-vs-Rust Monte Carlo performance comparison.

Run it from the project root so imports resolve cleanly.

In [ ]:
from pathlib import Path
import importlib
import importlib.util
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'heston_var').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def ensure_rust_backend_for_current_kernel():
    if importlib.util.find_spec('heston_var_rust') is not None:
        return True
    wheel_dir = PROJECT_ROOT / 'rust_engine' / 'target' / 'wheels'
    py_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
    wheels = sorted(wheel_dir.glob(f'heston_var_rust-*-{py_tag}-*.whl'))
    if not wheels:
        all_wheels = sorted(wheel_dir.glob('heston_var_rust-*.whl'))
        print(f'Rust backend is not installed for this notebook Python: {sys.executable}')
        print(f'This kernel needs a wheel tagged {py_tag}.')
        if all_wheels:
            print('Available Rust wheel(s):')
            for item in all_wheels:
                print(f'  - {item.name}')
            print('Fix: switch the notebook kernel to Python 3.11 (Heston VaR), or rebuild the Rust wheel with this kernel Python.')
        else:
            print('No Rust wheel was found. Build one with: cd rust_engine; python -m maturin build --release')
        return False
    wheel = wheels[-1]
    print(f'Installing Rust backend into this notebook kernel: {sys.executable}')
    print(f'Wheel: {wheel}')
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', '--force-reinstall', str(wheel)])
    except subprocess.CalledProcessError as exc:
        print(f'Rust backend install failed with exit code {exc.returncode}.')
        print('Fix: switch the notebook kernel to Python 3.11 (Heston VaR), then restart and run from the top.')
        return False
    importlib.invalidate_caches()
    return importlib.util.find_spec('heston_var_rust') is not None

ensure_rust_backend_for_current_kernel()

from heston_var.data import MarketDataConfig, YahooQueryMarketData, returns_from_prices
from heston_var.demo import synthetic_returns
from heston_var.heston import estimate_heston_params_frame
from heston_var.portfolio import Portfolio
from heston_var.risk import risk_from_pnl
import heston_var.rust_backend as rust_backend_module
rust_backend_module = importlib.reload(rust_backend_module)
rust_backend_available = rust_backend_module.rust_backend_available
rust_backend_info = getattr(
    rust_backend_module,
    'rust_backend_info',
    lambda: 'heston_var.rust_backend is an older in-memory module; restart the kernel.',
)
import heston_var.simulation as simulation_module
simulation_module = importlib.reload(simulation_module)
import heston_var.models as models_module
models_module = importlib.reload(models_module)
import heston_var.engine as engine_module
engine_module = importlib.reload(engine_module)
PortfolioVaREngine = engine_module.PortfolioVaREngine
heston_portfolio_var = simulation_module.heston_portfolio_var
simulate_heston_portfolio_pnl = simulation_module.simulate_heston_portfolio_pnl

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

print('Project root:', PROJECT_ROOT)
print('Notebook Python:', sys.executable)
print('Rust backend available:', rust_backend_available())
print('Rust backend info:', rust_backend_info())

## 1. Portfolio Data

The default path uses synthetic returns so the notebook is reproducible offline. Set `DATA_MODE = 'csv'` and point `PORTFOLIO_CSV` to any portfolio file with ticker/weight, market-value, or quantity/price columns.

In [ ]:
DATA_MODE = 'synthetic'  # choose: 'synthetic' or 'csv'
START_DATE = '2021-01-01'
PORTFOLIO_CSV = PROJECT_ROOT / 'configs' / 'large_multi_market_portfolio.csv'
INITIAL_CAPITAL = 1_000_000.0

if DATA_MODE == 'synthetic':
    returns = synthetic_returns(assets=4, observations=900, seed=7)
    portfolio = Portfolio.equal_weight(tuple(returns.columns), initial_capital=INITIAL_CAPITAL)
elif DATA_MODE == 'csv':
    portfolio = Portfolio.from_csv(PORTFOLIO_CSV, initial_capital=INITIAL_CAPITAL)
    loader = YahooQueryMarketData(MarketDataConfig(start=START_DATE))
    prices = loader.adjusted_close(portfolio.tickers)
    returns = returns_from_prices(prices)
else:
    raise ValueError("DATA_MODE must be one of: 'synthetic', 'csv'")

print('Data mode:', DATA_MODE)
print('Assets loaded:', len(portfolio.tickers))
print('Return observations:', len(returns))
display(pd.DataFrame({'ticker': portfolio.tickers, 'weight': portfolio.weights}).head(12))
returns.tail()

## 2. VaR Report

`backend='auto'` uses Rust when the extension is installed and falls back to NumPy otherwise. The first CSV run can take longer because market data for the ticker set is downloaded and cached.

In [ ]:
horizons = {'1-Day': 1, '1-Month': 21, '1-Year': 252}
reports = {}
risk_tables = []

for label, horizon_days in horizons.items():
    engine = PortfolioVaREngine(confidence=0.99, paths=50_000, horizon_days=horizon_days, seed=42, backend='auto')
    report = engine.run(portfolio, returns)
    reports[label] = report
    frame = report.to_frame()
    frame.insert(0, 'horizon', label)
    risk_tables.append(frame)

horizon_risk_table = pd.concat(risk_tables, ignore_index=True)
risk_table = horizon_risk_table[horizon_risk_table['horizon'].eq('1-Day')].drop(columns='horizon')

for label, report in reports.items():
    print(label, report.heston_diagnostics)
horizon_risk_table

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
plot_data = risk_table.set_index('model')[['var', 'expected_shortfall']]
plot_data.plot(kind='bar', ax=ax)
ax.set_title('VaR and Expected Shortfall by Model')
ax.set_ylabel('Dollars')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=0)
fig.tight_layout()

## 3. Visualize Monte Carlo Paths

The production engine stores terminal P&L for speed. For visualization, this helper simulates a small number of full price paths for one selected asset.

In [ ]:
def simulate_price_paths_for_plot(params, paths=30, horizon_days=60, seed=123, trading_days=252):
    rng = np.random.default_rng(seed)
    dt = 1.0 / trading_days
    s = np.ones((horizon_days + 1, paths))
    v = np.full(paths, max(params.v0, 1e-10))
    rho = np.clip(params.rho, -0.98, 0.98)

    for day in range(1, horizon_days + 1):
        z_price = rng.standard_normal(paths)
        z_extra = rng.standard_normal(paths)
        z_var = rho * z_price + np.sqrt(1.0 - rho**2) * z_extra
        v_pos = np.maximum(v, 0.0)
        v = v + params.kappa * (params.theta - v_pos) * dt + params.xi * np.sqrt(v_pos * dt) * z_var
        v = np.maximum(v, 1e-10)
        log_return = (params.mu - 0.5 * v_pos) * dt + np.sqrt(v_pos * dt) * z_price
        s[day] = s[day - 1] * np.exp(log_return)
    return s

params_by_asset = estimate_heston_params_frame(portfolio.align_returns(returns))
selected_asset = portfolio.tickers[0]
paths = simulate_price_paths_for_plot(params_by_asset[selected_asset], paths=30, horizon_days=60)

fig, ax = plt.subplots(figsize=(10, 5.2))
ax.plot(paths, linewidth=1.0, alpha=0.75)
ax.set_title(f'First 30 Heston Monte Carlo Price Paths: {selected_asset}')
ax.set_xlabel('Trading Day')
ax.set_ylabel('Relative Price')
fig.tight_layout()

## 4. Terminal P&L Distribution

This compares the simulated terminal portfolio P&L distribution with the historical daily P&L distribution.

In [ ]:
heston_result = heston_portfolio_var(
    portfolio,
    returns,
    confidence=0.99,
    paths=100_000,
    horizon_days=1,
    seed=42,
    backend='auto',
    antithetic=True,
)
historical_pnl = portfolio.historical_pnl(returns).to_numpy()

fig, ax = plt.subplots(figsize=(10, 5.2))
ax.hist(heston_result.pnl, bins=80, density=True, alpha=0.55, label=f'Heston MC ({heston_result.diagnostics.backend})')
ax.hist(historical_pnl, bins=50, density=True, alpha=0.45, label='Historical daily P&L')
ax.axvline(-heston_result.risk.var, color='tab:red', linestyle='--', label='Heston 99% VaR loss threshold')
ax.set_title('Terminal Portfolio P&L Distribution')
ax.set_xlabel('Portfolio P&L')
ax.set_ylabel('Density')
ax.legend()
fig.tight_layout()

heston_result.risk

## 5. Explicit Python vs Rust Monte Carlo Comparison

This section directly compares the NumPy simulator and the Rust/PyO3 simulator. It estimates Heston parameters once, reuses the same portfolio inputs, then reports runtime, speedup, and 99% VaR side by side.

Because Python and Rust use different random number generators, the VaR values are expected to be close but not identical. The runtime comparison is the main point of this section.

Note: the Rust backend is expected to win after the optimized Rayon build is installed. A scalar single-threaded Rust loop can be slower than NumPy because NumPy already runs compiled vectorized C loops.

If the first setup cell says `older build without backend_info`, restart the notebook kernel. Python cannot unload an already-imported compiled Rust extension from a live kernel.

In [ ]:
aligned_returns = portfolio.align_returns(returns)
params = estimate_heston_params_frame(aligned_returns)
rust_info = rust_backend_info()
rust_current = rust_backend_available() and 'rayon-parallel' in rust_info
print('Benchmark Rust backend info:', rust_info)
if rust_backend_available() and not rust_current:
    print('The loaded Rust extension is stale. Restart the kernel to load the rebuilt wheel before benchmarking Rust with antithetic variates.')

path_grid = [25_000, 100_000, 250_000, 1_000_000]
repeats = 3

def run_backend_once(backend, n_paths, seed, antithetic=True):
    start = time.perf_counter()
    pnl, backend_used = simulate_heston_portfolio_pnl(
        portfolio=portfolio,
        returns=aligned_returns,
        params=params,
        paths=n_paths,
        horizon_days=1,
        seed=seed,
        backend=backend,
        antithetic=antithetic,
    )
    elapsed = time.perf_counter() - start
    var_99 = risk_from_pnl(pnl, 0.99, backend_used).var
    return backend_used, elapsed, var_99

# Warm up imports, CPU caches, and the Rayon thread pool before timing.
for backend in ['python'] + (['rust'] if rust_current else []):
    run_backend_once(backend, 2_000, seed=999, antithetic=True)

rows = []
for n_paths in path_grid:
    python_times, python_vars = [], []
    rust_times, rust_vars = [], []

    for repeat in range(repeats):
        _, elapsed, var_99 = run_backend_once('python', n_paths, seed=10_000 + repeat)
        python_times.append(elapsed)
        python_vars.append(var_99)

        if rust_current:
            _, elapsed, var_99 = run_backend_once('rust', n_paths, seed=20_000 + repeat)
            rust_times.append(elapsed)
            rust_vars.append(var_99)

    row = {
        'paths': n_paths,
        'python_seconds': float(np.median(python_times)),
        'python_var_99': float(np.mean(python_vars)),
    }
    if rust_times:
        row.update({
            'rust_seconds': float(np.median(rust_times)),
            'rust_var_99': float(np.mean(rust_vars)),
            'python_over_rust_speedup': float(np.median(python_times) / np.median(rust_times)),
            'var_abs_diff': float(abs(np.mean(python_vars) - np.mean(rust_vars))),
        })
    rows.append(row)

comparison = pd.DataFrame(rows)
display(comparison)

if not rust_current:
    print('Current optimized Rust backend is not loaded, so only Python timings are shown.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

if 'rust_seconds' in comparison.columns:
    runtime_plot = comparison.set_index('paths')[['python_seconds', 'rust_seconds']]
    runtime_plot.plot(kind='bar', ax=axes[0])
    axes[0].set_title('Runtime: Python vs Rust')
    axes[0].set_xlabel('Paths')
    axes[0].set_ylabel('Median seconds')
    axes[0].tick_params(axis='x', rotation=0)

    axes[1].bar(comparison['paths'].astype(str), comparison['python_over_rust_speedup'])
    axes[1].axhline(1.0, color='black', linewidth=1, linestyle='--')
    axes[1].set_title('Speedup: Python Time / Rust Time')
    axes[1].set_xlabel('Paths')
    axes[1].set_ylabel('Speedup multiple')
else:
    comparison.set_index('paths')[['python_seconds']].plot(kind='bar', ax=axes[0])
    axes[0].set_title('Python Runtime')
    axes[0].set_xlabel('Paths')
    axes[0].set_ylabel('Median seconds')
    axes[1].axis('off')

fig.tight_layout()

### Antithetic Variates Check

This compares the Rust backend with antithetic variates enabled and disabled for the same requested number of output P&L samples. Antithetic mode generates paired shocks internally, so it can improve statistical efficiency and reduce independent random draws.

In [ ]:
if rust_current:
    rows = []
    for antithetic in [True, False]:
        timings = []
        vars_99 = []
        for repeat in range(5):
            backend_used, elapsed, var_99 = run_backend_once(
                'rust',
                250_000,
                seed=30_000 + repeat,
                antithetic=antithetic,
            )
            timings.append(elapsed)
            vars_99.append(var_99)
        rows.append({
            'backend': backend_used,
            'antithetic': antithetic,
            'paths': 250_000,
            'median_seconds': float(np.median(timings)),
            'mean_var_99': float(np.mean(vars_99)),
            'var_99_std_across_repeats': float(np.std(vars_99, ddof=1)),
        })
    antithetic_comparison = pd.DataFrame(rows)
    display(antithetic_comparison)
else:
    print('Current optimized Rust backend is not loaded. Restart the kernel, then rerun from the top.')